# Bloque II — Regresión y comparación de modelos

**Duración estimada:** 3 horas  
**Dataset:** `../data/ventas_mayo_2026.csv`

## Objetivo de aprendizaje

El alumnado aprenderá a construir modelos de regresión para predecir una variable numérica continua, comparar distintos algoritmos y justificar la elección del modelo mediante métricas.

## Agenda de 3 horas

| Tiempo | Actividad |
|---:|---|
| 0:00–0:25 | Qué es un problema de regresión |
| 0:25–0:55 | Preparación del dataset |
| 0:55–1:25 | Regresión lineal simple y múltiple |
| 1:25–1:35 | Pausa |
| 1:35–2:05 | Ridge, Lasso y Random Forest |
| 2:05–2:35 | Métricas y comparación |
| 2:35–3:00 | Caso práctico |

In [ ]:
# Configuración común
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 1. Carga y preparación inicial

La variable objetivo será `importe`, que representa el valor de la operación. El objetivo del modelo será estimar dicho importe a partir de variables como unidades, precio, descuento, canal, categoría, región y antigüedad del cliente.

In [ ]:
df = pd.read_csv("../data/ventas_mayo_2026.csv")
df = df.drop_duplicates()
df["fecha"] = pd.to_datetime(df["fecha"])
df["mes"] = df["fecha"].dt.month
df.head()

## 2. Definición de X e y

En regresión:

- `X` contiene las variables predictoras.
- `y` contiene la variable numérica que queremos predecir.

In [ ]:
target = "importe"

features_num = ["unidades", "precio_unitario", "descuento", "antiguedad_cliente_meses", "mes"]
features_cat = ["categoria", "region", "canal"]

X = df[features_num + features_cat]
y = df[target]

X.head()

## 3. División train/test

La evaluación debe hacerse con datos no usados durante el entrenamiento. Usamos un 80 % para entrenamiento y un 20 % para prueba.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape)

## 4. Preprocesamiento

Los modelos necesitan datos numéricos. Las variables categóricas se codifican con One-Hot Encoding y las numéricas se imputan y escalan cuando procede.

Usaremos `Pipeline` para evitar fugas de información y mantener el flujo reproducible.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, features_num),
        ("cat", categorical_transformer, features_cat)
    ]
)

In [ ]:
def make_preprocessor(features_num, features_cat):
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])
    return ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, features_num),
            ("cat", categorical_transformer, features_cat)
        ]
    )


## 5. Función de evaluación

Centralizamos las métricas para comparar todos los modelos con el mismo criterio.

- MAE: error absoluto medio.
- RMSE: penaliza errores grandes.
- R²: proporción de variabilidad explicada.

In [ ]:
def evaluar_modelo(nombre, modelo, X_train, X_test, y_train, y_test):
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)

    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)

    return {
        "modelo": nombre,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }, pred

## 6. Regresión lineal

La regresión lineal es interpretable y sirve como primera referencia. Aunque no siempre será el modelo más preciso, es muy útil para establecer una línea base.

In [ ]:
modelo_lr = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

res_lr, pred_lr = evaluar_modelo("Linear Regression", modelo_lr, X_train, X_test, y_train, y_test)
res_lr

## 7. Ridge y Lasso

Ridge y Lasso introducen regularización. Esto reduce el riesgo de sobreajuste y ayuda a controlar la complejidad del modelo.

In [ ]:
modelo_ridge = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", Ridge(alpha=1.0))
])

modelo_lasso = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", Lasso(alpha=0.05, max_iter=10000))
])

res_ridge, pred_ridge = evaluar_modelo("Ridge", modelo_ridge, X_train, X_test, y_train, y_test)
res_lasso, pred_lasso = evaluar_modelo("Lasso", modelo_lasso, X_train, X_test, y_train, y_test)

res_ridge, res_lasso

## 8. Random Forest Regressor

Random Forest permite capturar relaciones no lineales e interacciones entre variables. Suele mejorar el rendimiento, aunque a costa de menor interpretabilidad.

In [ ]:
modelo_rf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        max_depth=None
    ))
])

res_rf, pred_rf = evaluar_modelo("Random Forest", modelo_rf, X_train, X_test, y_train, y_test)
res_rf

## 9. Comparación de modelos

La decisión final debe combinar precisión, interpretabilidad, estabilidad y facilidad de despliegue.

In [ ]:
resultados = pd.DataFrame([res_lr, res_ridge, res_lasso, res_rf])
resultados.sort_values("RMSE")

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(resultados["modelo"], resultados["RMSE"])
plt.title("Comparación de modelos por RMSE")
plt.ylabel("RMSE")
plt.xticks(rotation=25)
plt.show()

## 10. Real vs predicho

Este gráfico permite observar si el modelo tiende a infraestimar o sobreestimar.

In [ ]:
mejor_pred = pred_rf

plt.figure(figsize=(5, 5))
plt.scatter(y_test, mejor_pred, alpha=0.7)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()])
plt.xlabel("Valor real")
plt.ylabel("Predicción")
plt.title("Real vs predicho - Random Forest")
plt.show()

## 11. Ejercicio integrador

1. Cambia el porcentaje de test a 30 % y compara resultados.
2. Modifica `max_depth` en Random Forest.
3. Añade o elimina variables predictoras.
4. Determina qué modelo seleccionarías y justifica por qué.
5. Redacta una conclusión de negocio.

### Pregunta de cierre

¿Elegirías siempre el modelo con menor RMSE? Justifica tu respuesta.

In [ ]:
# Ejercicio integrador: test 30%, max_depth y selección de variables

# 1. Cambiamos el tamaño del conjunto de test a 30 %.
X_train_30, X_test_30, y_train_30, y_test_30 = train_test_split(
    X, y, test_size=0.30, random_state=42
)
print('Train:', X_train_30.shape, 'Test:', X_test_30.shape)

# 2. Evaluamos los mismos modelos con el nuevo split.
modelos_30 = [
    ('Linear Regression', Pipeline([
        ('preprocessor', make_preprocessor(features_num, features_cat)),
        ('model', LinearRegression())
    ])),
    ('Ridge', Pipeline([
        ('preprocessor', make_preprocessor(features_num, features_cat)),
        ('model', Ridge(alpha=1.0))
    ])),
    ('Lasso', Pipeline([
        ('preprocessor', make_preprocessor(features_num, features_cat)),
        ('model', Lasso(alpha=0.05, max_iter=10000))
    ])),
    ('Random Forest', Pipeline([
        ('preprocessor', make_preprocessor(features_num, features_cat)),
        ('model', RandomForestRegressor(n_estimators=200, random_state=42, max_depth=None))
    ]))
]

resultados_30 = []
for nombre, modelo in modelos_30:
    res, _ = evaluar_modelo(nombre, modelo, X_train_30, X_test_30, y_train_30, y_test_30)
    resultados_30.append(res)

resultados_30 = pd.DataFrame(resultados_30).sort_values('RMSE')
resultados_30

# 3. Probamos un Random Forest con max_depth=6 para controlar complejidad.
modelo_rf_6 = Pipeline(steps=[
    ('preprocessor', make_preprocessor(features_num, features_cat)),
    ('model', RandomForestRegressor(n_estimators=200, random_state=42, max_depth=6))
])

res_rf_6, pred_rf_6 = evaluar_modelo(
    'Random Forest max_depth=6', modelo_rf_6, X_train_30, X_test_30, y_train_30, y_test_30
)

# 4. Probamos un conjunto de variables reducido para ver si el modelo se mantiene estable.
features_reducidos = [
    'unidades', 'precio_unitario', 'antiguedad_cliente_meses',
    'mes', 'categoria', 'region', 'canal'
]

X_reducido = df[features_reducidos]
X_train_red, X_test_red, y_train_red, y_test_red = train_test_split(
    X_reducido, y, test_size=0.30, random_state=42
)

modelo_rf_red = Pipeline(steps=[
    ('preprocessor', make_preprocessor(
        ['unidades', 'precio_unitario', 'antiguedad_cliente_meses', 'mes'],
        ['categoria', 'region', 'canal']
    )),
    ('model', RandomForestRegressor(n_estimators=200, random_state=42, max_depth=6))
])

res_rf_red, pred_rf_red = evaluar_modelo(
    'Random Forest reducido', modelo_rf_red, X_train_red, X_test_red, y_train_red, y_test_red
)

comparacion = pd.DataFrame([res_rf_6, res_rf_red]).sort_values('RMSE')
print('Comparación de Random Forest con diferentes configuraciones:')
comparacion

# 5. Conclusión de negocio
print('Conclusión:')
print('1. Cambiar el test a 30 % permite una estimación más conservadora del desempeño en datos nuevos.')
print('2. Random Forest con max_depth=6 reduce la complejidad sin perder precisión de forma importante.')
print('3. El modelo con variables reducidas es útil si se busca simplificar la implementación.')
print('4. No elegiría siempre el modelo con menor RMSE: también debe considerarse interpretabilidad, estabilidad y facilidad de despliegue, especialmente si la mejora es pequeña.')
